# Dynamic Prompt
<img src="./assets/LC_DynamicPrompts.png" width="500">

## Setup

Load and/or check for needed environmental variables

In [1]:
from dotenv import load_dotenv
from env_utils import doublecheck_env

# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env("example.env")

CRUSOE_API_KEY=****bmsK
LANGSMITH_API_KEY=****here
LANGSMITH_TRACING=false
LANGSMITH_PROJECT=****ials


In [2]:
from sql_db import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///Chinook.db")

In [3]:
from dataclasses import dataclass


@dataclass
class RuntimeContext:
    is_employee: bool
    db: SQLDatabase

In [4]:
from langchain_core.tools import tool
from langgraph.runtime import get_runtime

@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite command and return results."""
    runtime = get_runtime(RuntimeContext)
    db = runtime.context.db

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [5]:
SYSTEM_PROMPT_TEMPLATE = """You are a careful SQLite analyst.

Rules:
- Think step-by-step.
- When you need data, call the tool `execute_sql` with ONE SELECT query.
- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.
- Limit to 5 rows unless the user explicitly asks otherwise.
{table_limits}
- If the tool returns 'Error:', revise the SQL and try again.
- Prefer explicit column lists; avoid SELECT *.
"""

## Build a Dynamic Prompt
Utilize runtime context and middleware to generate a dynamic prompt.

In [6]:
from langchain.agents.middleware.types import ModelRequest, dynamic_prompt


@dynamic_prompt
def dynamic_system_prompt(request: ModelRequest) -> str:
    if not request.runtime.context.is_employee:
        table_limits = "- Limit access to these tables: Album, Artist, Genre, Playlist, PlaylistTrack, Track."
    else:
        table_limits = ""

    return SYSTEM_PROMPT_TEMPLATE.format(table_limits=table_limits)

Include middleware in `create_agent`.

In [7]:
from langchain_crusoe import ChatCrusoe
from langchain.agents import create_agent

agent = create_agent(
    model=ChatCrusoe(model="zai/GLM-5.2"),
    tools=[execute_sql],
    middleware=[dynamic_system_prompt],
    context_schema=RuntimeContext,
)

In [8]:
question = "What is the most costly purchase by Frank Harris?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    context=RuntimeContext(is_employee=False, db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What is the most costly purchase by Frank Harris?


================================== Ai Message ==================================

I need to find information about purchases and a person named Frank Harris. Let me first check what's available in the tables I have access to. Let me look for Frank Harris and also examine the schema of relevant tables.
Tool Calls:
  execute_sql (chatcmpl-tool-8887a7d111bb9a5b)
 Call ID: chatcmpl-tool-8887a7d111bb9a5b
  Args:
    query: SELECT ArtistId, Name FROM Artist WHERE Name LIKE '%Frank%' OR Name LIKE '%Harris%' LIMIT 5
  execute_sql (chatcmpl-tool-822d27510b7cc43d)
 Call ID: chatcmpl-tool-822d27510b7cc43d
  Args:
    query: SELECT sql FROM sqlite_master WHERE type='table' AND name IN ('Album','Artist','Genre','Playlist','PlaylistTrack','Track')
================================= Tool Message =================================
Name: execute_sql

[('CREATE TABLE [Album]\n(\n    [AlbumId] INTEGER  NOT NULL,\n    [Title] NVARCHAR(160)  NOT NULL,\n    [ArtistId] INTEGER  NOT NULL,\n    CONSTRAINT [PK_Al

================================== Ai Message ==================================

I have access to tables about music metadata (Album, Artist, Genre, Playlist, PlaylistTrack, Track), but none of these tables contain customer or purchase/invoice information. The Track table has a `UnitPrice` field, but there's no table linking purchases to specific customers.

Let me check if "Frank Harris" might appear as a composer in the Track table:
Tool Calls:
  execute_sql (chatcmpl-tool-94e2e7a41d02e151)
 Call ID: chatcmpl-tool-94e2e7a41d02e151
  Args:
    query: SELECT TrackId, Name, Composer, UnitPrice FROM Track WHERE Composer LIKE '%Frank Harris%' LIMIT 5
================================= Tool Message =================================
Name: execute_sql




================================== Ai Message ==================================

No results. There is no "Frank Harris" listed as a composer either.

I should note that the tables I have access to — **Album, Artist, Genre, Playlist, PlaylistTrack, and Track** — contain music metadata only (artists, albums, tracks, genres, and playlists). These tables do **not** include any customer or purchase/invoice data.

The standard Chinook database (which these tables belong to) also contains `Customer`, `Invoice`, and `InvoiceLine` tables that would hold purchase information, but I'm restricted from accessing those tables.

**Conclusion:** I'm unable to determine the most costly purchase by Frank Harris with the tables currently available to me. If access to the `Customer`, `Invoice`, and `InvoiceLine` tables can be granted, I'd be happy to find the answer!


In [9]:
question = "What is the most costly purchase by Frank Harris?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    context=RuntimeContext(is_employee=True, db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What is the most costly purchase by Frank Harris?


================================== Ai Message ==================================

I'll start by exploring the database schema to understand the structure.
Tool Calls:
  execute_sql (chatcmpl-tool-8ba3ed0352769f26)
 Call ID: chatcmpl-tool-8ba3ed0352769f26
  Args:
    query: SELECT sql FROM sqlite_master WHERE type='table'
================================= Tool Message =================================
Name: execute_sql

[('CREATE TABLE [Album]\n(\n    [AlbumId] INTEGER  NOT NULL,\n    [Title] NVARCHAR(160)  NOT NULL,\n    [ArtistId] INTEGER  NOT NULL,\n    CONSTRAINT [PK_Album] PRIMARY KEY  ([AlbumId]),\n    FOREIGN KEY ([ArtistId]) REFERENCES [Artist] ([ArtistId]) \n\t\tON DELETE NO ACTION ON UPDATE NO ACTION\n)',), ('CREATE TABLE [Artist]\n(\n    [ArtistId] INTEGER  NOT NULL,\n    [Name] NVARCHAR(120),\n    CONSTRAINT [PK_Artist] PRIMARY KEY  ([ArtistId])\n)',), ('CREATE TABLE [Customer]\n(\n    [CustomerId] INTEGER  NOT NULL,\n    [FirstName] NVARCHAR(40)  NOT NULL,\n    [LastName] N

================================== Ai Message ==================================

This is the Chinook music database. I need to find Frank Harris's invoices and identify the one with the highest total. Let me query the `Invoice` and `Customer` tables.
Tool Calls:
  execute_sql (chatcmpl-tool-bf49766d182f7bdb)
 Call ID: chatcmpl-tool-bf49766d182f7bdb
  Args:
    query: 
SELECT i.InvoiceId, i.InvoiceDate, i.Total, i.BillingCountry,
       c.FirstName, c.LastName
FROM Invoice i
JOIN Customer c ON i.CustomerId = c.CustomerId
WHERE c.FirstName = 'Frank' AND c.LastName = 'Harris'
ORDER BY i.Total DESC
LIMIT 5
================================= Tool Message =================================
Name: execute_sql

[(145, '2010-09-23 00:00:00', 13.86, 'USA', 'Frank', 'Harris'), (200, '2011-05-24 00:00:00', 8.91, 'USA', 'Frank', 'Harris'), (374, '2013-07-04 00:00:00', 5.94, 'USA', 'Frank', 'Harris'), (352, '2013-04-01 00:00:00', 3.96, 'USA', 'Frank', 'Harris'), (134, '2010-08-13 00:00:00', 1.98, 'USA

================================== Ai Message ==================================

Now let me get the details of that most costly invoice to show what was purchased.
Tool Calls:
  execute_sql (chatcmpl-tool-a358596ca3630209)
 Call ID: chatcmpl-tool-a358596ca3630209
  Args:
    query: 
SELECT il.InvoiceLineId, t.Name AS TrackName, a.Title AS AlbumName, ar.Name AS ArtistName,
       il.UnitPrice, il.Quantity, (il.UnitPrice * il.Quantity) AS LineTotal
FROM InvoiceLine il
JOIN Invoice i ON il.InvoiceId = i.InvoiceId
JOIN Track t ON il.TrackId = t.TrackId
JOIN Album a ON t.AlbumId = a.AlbumId
JOIN Artist ar ON a.ArtistId = ar.ArtistId
WHERE i.InvoiceId = 145
ORDER BY LineTotal DESC
================================= Tool Message =================================
Name: execute_sql

[(782, 'Ghost Of The Navigator', 'Brave New World', 'Iron Maiden', 0.99, 1, 0.99), (783, 'Wildest Dreams', 'Dance Of Death', 'Iron Maiden', 0.99, 1, 0.99), (784, 'Age Of Innocence', 'Dance Of Death', 'Iron Maiden', 

================================== Ai Message ==================================

Here's the answer:

## Frank Harris's Most Costly Purchase

**Invoice #145** — dated **September 23, 2010** — with a total of **$13.86**.

The purchase consisted of **14 tracks** (each priced at $0.99), all by the band **Iron Maiden**, spanning multiple albums:

| Track | Album |
|---|---|
| Ghost Of The Navigator | Brave New World |
| Wildest Dreams | Dance Of Death |
| Age Of Innocence | Dance Of Death |
| Chains Of Misery | Fear Of The Dark |
| Phantom of the Opera | Iron Maiden |
| Genghis Khan | Killers |
| The Trooper | Live After Death |
| Running Free | Live After Death |
| From Here To Eternity | Live At Donington 1992 (Disc 1) |
| Heaven Can Wait | Live At Donington 1992 (Disc 2) |
| Holy Smoke | No Prayer For The Dying |
| Where Eagles Dare | Piece Of Mind |
| Aces High | Powerslave |
| The Wicker Man | Rock In Rio [CD1] |

**Total: $13.86** (14 tracks × $0.99 each)
